# Autonomous Incident Response System for AWS
---

In this example, we will build an Agentic system to respond to incidents in your AWS accounts. This is a multi-agent system that composes 4 main components: 

1. **Monitoring**: This composes of a couple of aspects which includes monitoring CloudWatch alarms on the pre-built alarms that have already been set in your account. This might include high `CPU` usage, unhealthy load balancers, SageMaker instance cost allocations, etc. This would also include observing logs from different services from your account and classifying those logs into `Critical` (for example service down, `CPU`>`90%`), `Warning` (for example, latency > threshold, or if something goes beyond a threshold for a specific service) and `Informational` (for example, routine backups, information on various running applications in the AWS account, etc.).

1. **Diagnosis**: This includes diagnosis events that are seen through the monitoring agent. This can include querying `AWS` CloudTrail for additional data, X-Ray data and document these findings in reports that can be saved and used later in the resolution process. This would contain information only on the errors and the different services that need a resolution.

1. **Resolution**: This portion of the solution will be triggered by a diagnosis done from the step before. Once the diagnoses is done with the clear report, then this portion starts to remediate certain actions, such as adjusting EC2 auto-scaling group capacities, invoking functions to rollback deployments, etc. This agent is an essential part of the system since it will be using AWS `API`s in real time to manage the resources.

1. **Communication**: Last, this agent is responsible for keeping track of updates, creating and updating tickets in Jira, sending real time notifications to Slack with the incident details and resolution updates.

This solution will also contain aspects for observabilitiy and tracing but without further ado, let's get right into it.

In [1]:
# LangGraph is a low level orchestration framework for building controllable agents. 
# While langchain provides integrations and composable components to streamline LLM application development, 
# the LangGraph library enables agent orchestration, long term memory, human in the loop and customizable architectures.

In [2]:
import boto3
import logging
from datetime import datetime, timedelta
from typing import Annotated, List, Dict, Any
from typing_extensions import TypedDict
# import langgraph relevant libraries
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Import the memory saver to save in checkpoint and in some thread to retain agent's memory
from langgraph.checkpoint.memory import MemorySaver

# langchain imports
from langchain_aws.chat_models import ChatBedrockConverse
from langchain_core.tools import tool

In [ ]:
import os
from colorama import init, Fore, Style
import logging

# Initialize colorama
init()

# Create a custom formatter
class ColoredFormatter(logging.Formatter):
    def format(self, record):
        message = record.msg
        if isinstance(message, list):
            # Process each message in the list
            formatted_messages = []
            for msg in message:
                if msg.__class__.__name__ == 'HumanMessage':
                    formatted_msg = f"{Fore.GREEN}[Human] {msg.content}{Style.RESET_ALL}"
                elif msg.__class__.__name__ == 'AIMessage':
                    formatted_msg = f"{Fore.BLUE}[AI] {msg.content}{Style.RESET_ALL}"
                elif msg.__class__.__name__ == 'ToolMessage':
                    formatted_msg = f"{Fore.YELLOW}[Tool] {msg.content}...{Style.RESET_ALL}"
                else:
                    formatted_msg = str(msg)
                formatted_messages.append(formatted_msg)
            record.msg = '\n'.join(formatted_messages)
        return super().format(record)
# Set up logger with the custom formatter
logger = logging.getLogger(__name__)
handler = logging.StreamHandler()
handler.setFormatter(ColoredFormatter('%(message)s'))
logger.addHandler(handler)
logger.setLevel(logging.INFO)

In [7]:
# define the constants
AMAZON_NOVA_PRO_MODEL_ID: str = 'us.amazon.nova-pro-v1:0'

### State definitions
---

First, we will define the state for our sub agents: for Monitoring, Diagnosis, Remediation and the Supervisor. Since all of these will have similar states, let's go ahead and define a common `incidentState`.

In [8]:
from typing import List, Optional

class IncidentState(TypedDict):
    """
    A TypedDict class representing the state of monitoring.
    """
    # This tracks the user messages in the history and will be 
    # used to check for which is the next node/sub agent to use in this
    # agentic architecture
    messages: List[Dict]
    # This contains information on the alarm statuses in the AWS account
    alarms: Optional[List[Dict]]
    # This contains information on the metrics in the AWS account
    metrics: Optional[Dict]
    # Instance ids of your EC2 instances
    instance_ids: Optional[List[str]]
    # This contains information on the diagnosis that can be done to remediate
    diagnosis_report: Optional[str]
    remediation_actions: Optional[List[str]]
    notification_status: Optional[str]

With the help of this unified state, this does as follows:

1. **Ensures consistency**: Each agent in this case works with a consistent structure.

1. **Ease of communication**: This facilitates simpler data passing between nodes.

1. **Traceability**: Incident lifecycle remains centralized.

In [9]:
from langchain_core.messages import HumanMessage

llm = ChatBedrockConverse(
    model_id = AMAZON_NOVA_PRO_MODEL_ID, 
    temperature = 0.1,
)

#### Define monitoring tools
---

In [18]:
import boto3
from datetime import timedelta
from langchain_core.tools import BaseTool, tool
from langgraph.prebuilt import create_react_agent

# define some boto3 and AWS clients
cloudwatch_client = boto3.client('cloudwatch')
cloudtrail_client = boto3.client('cloudtrail')
xray_client = boto3.client('xray')
autoscaling_client = boto3.client('autoscaling')
ec2_client = boto3.client('ec2')

@tool
def fetch_ec2_metrics_for_alarm_instances(
    alarm_state: Annotated[str, "State of alarms to check, e.g., 'ALARM' or 'OK'"] = "ALARM",
    period: int = 300
) -> Optional[Dict[str, Dict[str, Optional[float]]]]:
    """
    Fetches alarms once and retrieves key utilization metrics for affected EC2 instances.
    Returns a dictionary mapping instance IDs to their metrics summary.
    """
    try:
        # This gets the alarm response from the cloudwatch client. This in this example can be EC2
        # instances that cross certain thresholds in your AWS account
        alarm_response = cloudwatch_client.describe_alarms(StateValue=alarm_state, MaxRecords=100)
    except Exception as e:
        print(f"Error fetching alarms: {e}")
        return None
    if not alarm_response or not alarm_response.get("MetricAlarms"):
        print(f"No alarms found in state: {alarm_state}")
        return {}
    # Identify affected EC2 instance IDs from alarms
    alarm_instances = {dim["Value"]
                       for alarm in alarm_response["MetricAlarms"]
                       for dim in alarm.get("Dimensions", [])
                       if dim["Name"] == "InstanceId"}
    # metrics of interest to fetch, using these metrics we will be able to notify the 
    # user via slack or create a ticket on jira for them to view and track
    metrics_to_fetch = [
        "CPUUtilization", "NetworkIn", "NetworkOut",
        "DiskReadOps", "DiskWriteOps", "StatusCheckFailed"
    ]
    instance_metrics_summary = {}
    # Fetch metrics for each instance
    for instance_id in alarm_instances:
        instance_summary = {}
        for metric_name in metrics_to_fetch:
            try:
                response = cloudwatch_client.get_metric_statistics(
                    Namespace="AWS/EC2",
                    MetricName=metric_name,
                    Dimensions=[{'Name': 'InstanceId', 'Value': instance_id}],
                    Period=period,
                    Statistics=['Average'],
                    StartTime=datetime.utcnow() - timedelta(minutes=10),
                    EndTime=datetime.utcnow()
                )
                datapoints = response.get("Datapoints", [])
                if datapoints:
                    # Sort datapoints by timestamp to pick latest
                    datapoints.sort(key=lambda x: x['Timestamp'], reverse=True)
                    instance_summary[metric_name] = datapoints[0]["Average"]
                else:
                    instance_summary[metric_name] = None
            except Exception as e:
                print(f"Error fetching metric '{metric_name}' for {instance_id}: {e}")
                instance_summary[metric_name] = None
        instance_metrics_summary[instance_id] = instance_summary
    return instance_metrics_summary

monitoring_toolkit = [fetch_ec2_metrics_for_alarm_instances]

In [19]:
# create the monitoring agent
monitoring_agent = create_react_agent(llm, tools=monitoring_toolkit, prompt="This agent is used to monitor AWS alarms and classify incidents within it as Critical, Warning or Informational.")
logger.info(f"Created the monitoring agent: {monitoring_agent}")

Created the monitoring agent: <langgraph.graph.state.CompiledStateGraph object at 0x1077b5640>
Created the monitoring agent: <langgraph.graph.state.CompiledStateGraph object at 0x1077b5640>
Created the monitoring agent: <langgraph.graph.state.CompiledStateGraph object at 0x1077b5640>


In [20]:
# Create the node for the monitoring agent
def monitoring_node(state: IncidentState):
    result = monitoring_agent.invoke(state)
    updated_state = {
        "messages": state["messages"] + [{"content": result["messages"][-1].content, "role": "monitoring"}],
        "alarms": result.get("alarms"),
        "metrics": result.get("metrics"),
        "instance_ids": result.get("instance_ids")
    }
    return updated_state

#### Define diagnosis tools
---

In [21]:
# next, we define some diagnosis tools, which includes running some diagnosis on cloudtrail events, and xray traces
from datetime import timedelta

@tool
def get_info_on_cloudtrail_events(event_name: Annotated[str, "Name of the CloudTrail event"], minutes: int = 60) -> Optional[Dict]:
    """
    Fetches the cloudtrail events for diagnosis
    """
    events = cloudtrail_client.lookup_events(
        LookupAttributes=[
            {
                'AttributeKey': 'EventName',
                'AttributeValue': event_name
            },
        ],
        StartTime=datetime.utcnow() - timedelta(minutes=minutes),
        EndTime=datetime.utcnow(),
        MaxResults=50
    )
    return events

@tool
def get_xray_traces(minutes: int = 60) -> Optional[Dict]:
    """
    Fetches the xray traces for diagnosis
    """
    traces = xray_client.get_trace_summaries(
        StartTime=datetime.utcnow() - timedelta(minutes=minutes),
        EndTime=datetime.utcnow(),
        Sampling=False
    )
    return traces['TraceSummaries']

diagnosis_toolkit = [get_info_on_cloudtrail_events, get_xray_traces]

In [22]:
# Next, let's bind these tools to our diagnosis agent
diagnosis_agent = create_react_agent(llm, tools=diagnosis_toolkit, prompt="Diagnose AWS incidents using CloudTrail, XRay data and document the findings")

### Test the monitoring and diagnosis agents
---

Next, after we have defined our monitoring and diagnosis tools and created the agents, we can invoke to test how they work.

In [ ]:
from langchain_core.messages import HumanMessage
content = """
I want to check the current alarms in my AWS account.
Then, summarize EC2 instance utilization metrics for me. 
"""

# Invoke monitoring agent with the given message
monitoring_response = monitoring_agent.invoke({"messages": [HumanMessage(content=content)]})
logger.info(monitoring_response["messages"])


/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_25944/4156480986.py:55: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  StartTime=datetime.utcnow() - timedelta(minutes=10),
/var/folders/jy/g9mb5j5n6c11fgdj788p5rww0000gr/T/ipykernel_25944/4156480986.py:56: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  EndTime=datetime.utcnow()
[Human] 
I want to check the current alarms in my AWS account.
Then, summarize EC2 instance utilization metrics for me. 

[AI] [{'type': 'text', 'text': "<thinking> To fulfill the user's request, I need to fetch the current alarms in the AWS account and then summarize the EC2 instance utilization metrics for the affected instances. The tool `f